#Proyecto: Control de facturación, cobros y morosidad

Este notebook crea una base de datos SQLite a partir de varios archivos CSV (clientes, productos, facturas, líneas y pagos).
Calcula totales de facturación, cobros, saldos, estado de cada factura y el DSO (Days Sales Outstanding).

In [17]:
# =====================================================
# 🔹 1. Clonar tu repositorio desde GitHub
# =====================================================
!rm -rf Portfolio-de-An-lisis-de-Datos  # borra la copia anterior si existía
!git clone https://github.com/carmenplata1106/Portfolio-de-An-lisis-de-Datos.git

# =====================================================
# 🔹 2. Definir rutas
# =====================================================
from pathlib import Path
BASE = Path("proyectos/proyectos/06_finanzas_facturacion_cobros/data")
DATA = BASE / "data"

print("📁 Carpeta base:", BASE)
print("📁 Carpeta data:", DATA)

# =====================================================
# 🔹 3. Verificar archivos CSV
# =====================================================
import glob, os
archivos = glob.glob(str(DATA / "*.csv"))
print("📊 Archivos CSV encontrados:")
for f in archivos:
    print(" -", os.path.basename(f))

# =====================================================
# 🔹 4. Cargar los CSV con pandas
# =====================================================
import pandas as pd

clientes  = pd.read_csv(DATA / "clientes.csv", parse_dates=["fecha_alta"])
productos = pd.read_csv(DATA / "productos.csv")
facturas  = pd.read_csv(DATA / "facturas.csv", parse_dates=["fecha_emision", "fecha_vencimiento"])
lineas    = pd.read_csv(DATA / "lineas_factura.csv")
pagos     = pd.read_csv(DATA / "pagos.csv", parse_dates=["fecha_pago"])

print("✅ Datos cargados correctamente")

# Verificamos tamaños
for nombre, df in [("clientes", clientes), ("productos", productos), ("facturas", facturas), ("lineas", lineas), ("pagos", pagos)]:
    print(f"{nombre:12}: {df.shape[0]} filas x {df.shape[1]} columnas")


Cloning into 'Portfolio-de-An-lisis-de-Datos'...
remote: Enumerating objects: 168, done.
remote: Counting objects: 100% (168/168), done.
remote: Compressing objects: 100% (152/152), done.
remote: Total 168 (delta 58), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (168/168), 6.70 MiB | 18.24 MiB/s, done.
Resolving deltas: 100% (58/58), done.
📁 Carpeta base: proyectos/proyectos/06_finanzas_facturacion_cobros/data
📁 Carpeta data: proyectos/proyectos/06_finanzas_facturacion_cobros/data/data
📊 Archivos CSV encontrados:


FileNotFoundError: [Errno 2] No such file or directory: 'proyectos/proyectos/06_finanzas_facturacion_cobros/data/data/clientes.csv'

In [10]:
#Importar librerías necesarias
import pandas as pd
from sqlalchemy import create_engine
from datetime import date
from pathlib import Path

In [11]:
#Definir rutas y conexión a la base de datos
BASE = Path('.')
DATA = BASE / 'data'
engine = create_engine(f'sqlite:///{BASE / 'finanzas.db'}')

print('Conectado a la base de datos:', BASE / 'finanzas.db')

Conectado a la base de datos: finanzas.db


In [12]:
#Cargar los archivos CSV
clientes = pd.read_csv(DATA / 'clientes.csv', parse_dates=['fecha_alta'])
productos = pd.read_csv(DATA / 'productos.csv')
facturas = pd.read_csv(DATA / 'facturas.csv', parse_dates=['fecha_emision', 'fecha_vencimiento'])
lineas = pd.read_csv(DATA / 'lineas_factura.csv')
pagos = pd.read_csv(DATA / 'pagos.csv', parse_dates=['fecha_pago'])

print('Datos cargados correctamente')

FileNotFoundError: [Errno 2] No such file or directory: 'data/clientes.csv'

In [ ]:
#Calcular totales por factura
lineas['importe_linea'] = lineas['cantidad'] * lineas['precio_unitario']
totales = lineas.groupby('factura_id', as_index=False)['importe_linea'].sum().rename(columns={'importe_linea': 'bruto'})
fact = facturas.merge(totales, on='factura_id', how='left').fillna({'bruto': 0})
fact['neto_sin_iva'] = fact['bruto'] * (1 - fact['descuento_pct'].fillna(0))
fact['total_con_iva'] = fact['neto_sin_iva'] * (1 + fact['iva_pct'].fillna(0))

In [ ]:
#Calcular cobros y saldos
pagos_factura = pagos.groupby('factura_id', as_index=False)['importe_pagado'].sum().rename(columns={'importe_pagado': 'cobrado'})
fact = fact.merge(pagos_factura, on='factura_id', how='left').fillna({'cobrado': 0})
fact['saldo'] = fact['total_con_iva'] - fact['cobrado']

In [ ]:
#Calcular estado de factura y DSO
today = pd.to_datetime(date.today())

def calcular_estado(f):
    if f['saldo'] <= 1e-6:
        return 'cobrada'
    elif f['cobrado'] > 0 and f['saldo'] > 0 and f['fecha_vencimiento'] >= today:
        return 'parcial'
    elif f['saldo'] > 0 and f['fecha_vencimiento'] < today:
        return 'vencida'
    else:
        return 'emitida'

fact['estado_calculado'] = fact.apply(calcular_estado, axis=1)

pagos_ordenados = pagos.sort_values(['factura_id', 'fecha_pago'])
primer_pago = pagos_ordenados.drop_duplicates('factura_id')[['factura_id', 'fecha_pago']].rename(columns={'fecha_pago': 'fecha_primer_pago'})
fact = fact.merge(primer_pago, on='factura_id', how='left')
fact['dso_dias'] = (fact['fecha_primer_pago'] - fact['fecha_emision']).dt.days

In [ ]:
#Guardar las tablas en la base de datos
clientes.to_sql('clientes', engine, if_exists='replace', index=False)
productos.to_sql('productos', engine, if_exists='replace', index=False)
lineas.to_sql('lineas_factura', engine, if_exists='replace', index=False)
pagos.to_sql('pagos', engine, if_exists='replace', index=False)
fact.to_sql('facturas', engine, if_exists='replace', index=False)
print('ETL completado. Se ha creado finanzas.db en la carpeta del proyecto.')